# The NDDataset object

The NDDataset is the main object used by **SpectroChemPy**.

Like numpy ndarrays, NDDatasets have the capability to be sliced, sorted and subjected to
mathematical operations.

But, in addition, NDDatasets may have units, can be masked and each dimension can have
coordinates also with units.
This makes NDDatasets aware of units compatibility, *e.g.*, for binary operations such as
addition or subtraction or during the application of mathematical operations.
In addition to or in replacement of numerical data for coordinates,
NDDatasets can also have labeled coordinates where labels can be different kinds of
objects (strings, datetime objects, numpy ndarrays or other NDDatasets, etc.).

This offers a lot of flexibility in using NDDatasets that, we hope, will be useful
for applications.
See the :ref:`examples-gallery` for
additional information about such possible applications.

## Table of Contents

* [Introduction](#Introduction)
  * [1D-Dataset (unidimensional dataset)](#1D-Dataset-(unidimensional-dataset))
  * [nD-Dataset (multidimensional dataset)](#nD-Dataset-(multidimensional-dataset))
* [Metadata and Attributes](#About-dates-and-times)
  * [About dates and times](#About-dates-and-times)
  * [About the history attribute](#About-the-history-attribute)
* [Units](#Units)
* [Coordinates](#Coordinates)
  * [Labels](#Labels)
  * [Sharing coordinates between dimensions](#Sharing-coordinates-between-dimensions)
  * [Setting coordinates using set_coordset](#Setting-coordinates-using-set_coordset)
  * [Adding several coordinates to a single dimension](#Adding-several-coordinates-to-a-single-dimension)
  * [Math operations on coordinates](#Math-operations-on-coordinates)
  * [Summary of coordinate setting syntax](#Summary-of-the-coordinate-setting-syntax)
* [Methods to create NDDataset](#Methods-to-create-NDDataset)
  * [Create a dataset from a function](#Create-a-dataset-from-a-function)
  * [Using numpy-like constructors](#Using-numpy-like-constructors-of-NDDatasets)
  * [Copying existing NDDataset](#Copying-existing-NDDataset)
  * [Importing from external datasets](#Importing-from-external-datasets)

## Introduction

**Below (and in the next sections), we try to give an almost complete view of the
NDDataset features.**

As we will make some reference to the
**[numpy](https://numpy.org/doc/stable/index.html)** library, we also import it here.

In [ ]:
import numpy as np

import spectrochempy as scp

We additionally import the three main SpectroChemPy objects that we will use through
this tutorial

In [ ]:
from spectrochempy import Coord
from spectrochempy import CoordSet
from spectrochempy import NDDataset

For a convenient usage of units, we will also directly import
**[ur](#Units)**, the unit registry which contains all available
units.

In [ ]:
from spectrochempy import ur

Multidimensional arrays are defined in Spectrochempy using the `NDDataset` object.

`NDDataset` objects mostly behave like numpy's `numpy.ndarray`
(see for instance
[numpy quickstart tutorial](https://numpy.org/doc/stable/user/quickstart.html)).

However, unlike raw numpy arrays, the presence of optional properties makes
them (hopefully) more appropriate for handling spectroscopic information,
which is one of the major objectives of the SpectroChemPy package:

* `mask`: Data can be partially masked at will
* `units`: Data can have units, allowing units-aware operations
* `CoordSet`: Data can have a set of coordinates, one or several per dimension

Additional metadata can also be added to the instances of this class
through the `meta` properties.

## 1D-Dataset (unidimensional dataset)

In the following example, a minimal 1D dataset is created from a simple list,
to which we can add some metadata:

In [ ]:
d1D = NDDataset(
    [10.0, 20.0, 30.0],
    name="Dataset N1",
    author="Blake and Mortimer",
    description="A dataset from scratch",
    history="creation",
)
d1D

In [ ]:
print(d1D)

In [ ]:
_ = d1D.plot(figsize=(3, 2))

Except few additional metadata such `author` , `created` ..., there is not much
difference with respect to a conventional
**[numpy.array](
https://numpy.org/doc/stable/reference/generated/numpy.array.html#numpy.array)**.
For example, one
can apply numpy
**[ufunc](https://numpy.org/doc/stable/reference/ufuncs.html#available-ufuncs)'s**
directly to a NDDataset or make basic arithmetic
operation with these objects:

In [ ]:
np.sqrt(d1D)

In [ ]:
d1D += d1D / 2.0
d1D

As seen above, there are some attributes that are automatically added to the dataset:

* `id`: This is a unique identifier for the object.
* `name`: A short and unique name for the dataset. It will be equal to the automatic
  `id` if it is not provided.
* `author`: Author determined from the computer name if not provided.
* `created`: Date and time of creation.
* `modified`: Date and time of modification.

These attributes can be modified by the user, but the `id`, `created` and `modified`
attributes are read only.

Some other attributes are defined to describe the data:
* `title`: A long name that will be used in plots or in some other operations.
* `history`: Timestamped entries for operations that participate in the history
  contract. It is inspectable but is not a complete provenance record.
* `description`: A comment or a description of the object's purpose or contents.
* `origin`: An optional reference to the source of the data.

Here is an example of the use of the NDDataset attributes:

In [ ]:
d1D.title = "intensity"
d1D.name = "mydataset"
d1D.history = "created from scratch"
d1D.description = "Some experimental measurements"
d1D

d1D is a 1D (1-dimensional) dataset with only one dimension.

Some attributes are useful to check this kind of information:

In [ ]:
d1D.shape  # the shape of 1D contain only one dimension size

In [ ]:
d1D.ndim  # the number of dimensions

In [ ]:
d1D.dims  # the name of the dimension (it has been automatically attributed)

**Note**: The names of the dimensions are set automatically. But they can be changed,
with <u>the limitation</u> that the
name must be a single letter.

In [ ]:
d1D.dims = ["q"]  # change the list of dim names.

In [ ]:
d1D.dims

### nD-Dataset (multidimensional dataset)

To create a nD NDDataset, we can provide a nD-array like object to the NDDataset
instance constructor

In [ ]:
d3D = NDDataset.random((2, 4, 6))
d3D.title = "energy"
d3D.author = "Someone"
d3D.name = "3D dataset creation"
d3D.history = "created from scratch"
d3D.description = "Some example"
d3D.dims = ["u", "v", "t"]
d3D

We can also add all information in a single statement

In [ ]:
d3D = NDDataset.random(
    (2, 4, 6),
    dims=["u", "v", "t"],
    title="Energy",
    author="Someone",
    name="3D_dataset",
    history="created from scratch",
    description="a single statement creation example",
)
d3D

Three names are attributed at creation (if they are not provided with the `dims`
attribute, then the names
'z','y','x' are automatically attributed)

In [ ]:
d3D.dims

In [ ]:
d3D.ndim

In [ ]:
d3D.shape

## About dates and times
The dates and times are stored internally as
[UTC (Coordinated Universal Time)](https://en.wikipedia.org/wiki/Coordinated_Universal_Time).
Timezone information is stored in the timezone attribute.
If not set, the default is to use the local timezone,
which is probably the most common case.

In [ ]:
nd = NDDataset()
nd.created

In this case our local timezone has been used by default for the conversion from
UTC datetime.

In [ ]:
nd.local_timezone

In [ ]:
nd.timezone = "EST"
nd.created

For a list of timezone code (TZ) you can have a look at
[List_of_tz_database_time_zones](
https://en.wikipedia.org/wiki/List_of_tz_database_time_zones).

## About the `history` attribute

An `NDDataset` owns one authoritative history store. The public ``history``
attribute renders it as a readable, timestamped, list-compatible view. The view
supports reading, slicing, display, iteration, and comparison, but is read-only:
direct mutations such as ``dataset.history.append(...)`` raise ``TypeError``.
Use ``annotate()``, ``replace_history()``, or ``clear_history()`` when a change is
intentional.

``history_entries`` provides the structured view as a deep, detached copy. Every
entry has exactly four fields:

- ``date``: the operation timestamp as a Python ``datetime``;
- ``operation``: a short identifier, or ``None`` for a text-only entry;
- ``parameters``: a detached dictionary of compact values;
- ``message``: the readable text rendered by ``history``.

Structured and text-only entries normally coexist. Reader messages, user
annotations, older histories, and treatments outside the structured coverage may
all have ``operation=None`` and an empty parameter dictionary.

This autonomous example starts with synthetic spectra and a user annotation, then
applies a structured shift and integrates along the same dimension. The integration
keeps the complete chronology that it receives.

In [ ]:
history_demo = NDDataset(
    np.array([[1.0, 2.0, 4.0, 8.0], [2.0, 3.0, 5.0, 9.0]]),
    dims=["y", "x"],
    coordset=[
        Coord([0.0, 1.0], title="sample"),
        Coord(
            [1000.0, 1100.0, 1200.0, 1300.0],
            units="cm^-1",
            title="wavenumber",
        ),
    ],
    name="history_demo",
)
history_demo.annotate("synthetic spectra prepared")
shifted = history_demo.roll(pts=1.9, dim="x")
integrated = shifted.trapezoid(dim="x")
integrated.history

The final entry names the integration kernel and records the requested dimension,
the resolved dimension, and the corresponding axis in its source dataset.
Parameter dictionaries are specific to each operation family; there is no universal
parameter schema.

In [ ]:
integration_entry = integrated.history_entries[-1]
integration_entry["operation"], integration_entry["parameters"]

A public alias can delegate to a differently named kernel, so structured entries
name the kernel that actually ran. When normalization, conversion, rounding, or
clamping makes an effective scientific parameter differ from the request, supported
families can retain both. Here the discrete shift applies one point even though
``1.9`` was requested.

In [ ]:
shift_entry = shifted.history_entries[-1]
(
    shift_entry["parameters"]["scientific_parameters"],
    shift_entry["parameters"]["requested_parameters"],
)

``history_entries`` is detached all the way through nested parameters. Changing the
returned copy does not modify the dataset. An explicit ``list(dataset.history)`` is
likewise an ordinary mutable copy of the readable strings.

In [ ]:
detached_entries = integrated.history_entries
detached_entries[-1]["parameters"]["resolved_axis"] = 99
integrated.history_entries[-1]["parameters"]["resolved_axis"]

Explicit history-management methods make destructive intent visible. Assigning a
string to ``history`` remains a compatibility shorthand for ``annotate()``; assigning
a list is a shorthand for ``replace_history()``; assigning ``None`` does nothing.
Structured mappings, legacy ``(date, message)`` pairs, and strings can be supplied to
``replace_history()``.

In [ ]:
editable_history = history_demo.copy()
editable_history.replace_history(["replacement note", "another note"])
editable_history.history

In [ ]:
editable_history.clear_history()
editable_history.history

Both shallow and deep dataset copies receive independent histories, including nested
parameter values. Supported native SCP/PSCP persistence preserves structured entries
with format version 3; xarray mapping and the NetCDF representation use version 2.
Current readers also accept native version 2 and portable version 1 textual histories,
retaining their dates and messages without inferring semantics from prose. Files
written with the newer structured formats are not guaranteed to be readable by older
SpectroChemPy versions. This is deliberately one-way compatibility.

For the forthcoming 1.1.0 release, structured entries cover the following bounded
families:

- dataset selection and slicing, and transposition;
- out-of-place scalar or ``NDDataset`` addition and subtraction;
- ``mean()``, ``sum()``, ``std()``, and ``var()`` when they return an ``NDDataset``;
- ``squeeze()``, ``swapdims()``, and ``reshape()``;
- point, circular, and Fourier shifts, and zero filling;
- apodization and phasing;
- successful ``mc()``, ``ps()``, ``ht()``, and ``dc()`` processing paths;
- ``trapezoid()`` and ``simpson()`` integration.

The recorded compact parameters depend on the family. Where its contract establishes
them, an entry may distinguish requested and effective scientific parameters, the
requested and resolved dimension, the axis in the source dataset, and execution
mode. These fields do not form a universal replay schema. Not every arithmetic
operation, reduction, or transformation is structured. In-place arithmetic remains
text-only, and ``snv()`` intentionally adds the single text entry
``SNVTransformer applied`` in both execution modes. This coverage is present on the
development branch for 1.1.0; it is not a claim about an already published stable
version.

This is a readable operation log, not exhaustive provenance. Several treatments
remain text-only; learned estimator state and every input chronology of a multi-source
assembly are not recorded. Histories are linear and separate, and they do not detect
arbitrary mutations, construct a provenance graph, or replay computations. The NMR
plugin processing trace and estimator-specific logs are separate mechanisms, not
``NDDataset.history``. Exact message wording is intended for people and is not a
stable machine-readable contract; inspect structured fields when available.

A short infrared workflow shows how the two views complement each other. We select
six spectra in the OH region, calculate a reference from the first three, and
subtract that reference from every selected spectrum.

In [ ]:
spectra = scp.read("irdata/nh4y-activation.spg")
region = spectra[:6, 3700.0:3300.0]
region.name = "OH-region"
reference = region[:3].mean(dim="y")
reference.name = "mean-reference"
corrected = region - reference
corrected.name = "referenced-OH-region"

In [ ]:
corrected.plot()

In [ ]:
reference.history

In [ ]:
[entry for entry in reference.history_entries if entry["operation"] == "mean"]

In [ ]:
corrected.history

In [ ]:
[
    entry
    for entry in corrected.history_entries
    if entry["operation"] in {"slice", "subtract"}
]

The reference log records that its mean was computed along ``y``. Its separate
chronology is not merged into ``corrected``: the final log follows the main dataset
and records its import, spectral selection, and subtraction. The subtraction entry
instead identifies the two operands in mathematical order. Understanding that the
subtracted reference is the mean of the first three spectra therefore requires its
own ``reference.history``; the corrected dataset alone does not contain that
construction. This keeps both logs linear and readable without presenting either
as complete provenance. The SPG import entry displays only
``nh4y-activation.spg`` so the log stays portable; the complete source path remains
available from ``spectra.filename`` and its reader metadata.

## Units

One interesting feature of NDDataset is the ability to define units for the internal
data.

In [ ]:
d1D.units = ur.eV  # ur is a registry containing all available units

In [ ]:
d1D  # note the eV symbol of the units added to the values field below

This allows to make units-aware calculations:

In [ ]:
d1D**2  # note the results in eV^2

In [ ]:
np.sqrt(d1D)  # note the result in e^0.5

In [ ]:
time = 5.0 * ur.second
d1D / time  # here we get results in eV/s

Compatible conversion returns a new dataset by default and leaves ``d1D`` unchanged.
Conversions such as energy to temperature or wavenumber to wavelength instead use
physical equivalence contexts; they are not ordinary same-dimensional rescaling.

In [ ]:
in_joules = d1D.to("J")
in_joules

In [ ]:
d1D  # still expressed in eV

## Coordinates

The above created `d3D` dataset has 3 dimensions, but no coordinates for these
dimensions. Here arises a big difference
with simple `numpy` arrays:
* We can add coordinates to each dimension of a NDDataset.

To get the list of all defined coordinates, we can use the `coords` attribute:

In [ ]:
d3D.coordset  # no coordinates, so it returns nothing (None)

In [ ]:
d3D.t  # the same for coordinate  t, v, u which are not yet set

To add coordinates, one way is to set them one by one:

In [ ]:
d3D.t = (
    Coord.arange(6) * 0.1
)  # we need a sequence of 6 values for `t` dimension (see shape above)
d3D.t.title = "time"
d3D.t.units = ur.seconds
d3D.coordset  # now return a list of coordinates

In [ ]:
d3D.t

In [ ]:
d3D.coordset("t")  # Alternative way to get a given coordinates

In [ ]:
d3D["t"]  # another alternative way to get a given coordinates

The two other coordinates u and v are still undefined

In [ ]:
d3D.u, d3D.v

When the dataset is printed, only the information for the existing coordinates is
given.

In [ ]:
d3D

Programmatically, we can use the attribute `is_empty` or `has_data` to check this

In [ ]:
d3D.v.has_data, d3D.v.is_empty

An error is raised when a coordinate doesn't exist

In [ ]:
try:
    d3D.x
except Exception as e:
    scp.error_(Exception, e)

In some case it can also be useful to get a coordinate from its title instead of its
name (the limitation is that if
several coordinates have the same title, then only the first ones that is found in
the coordinate list, will be
returned - this can be ambiguous)

In [ ]:
d3D["time"]

In [ ]:
d3D.time

## Labels

It is possible to use labels instead of numerical coordinates. Labels are sequences of
objects. The length of the sequence must be equal to the size of the dimension.

In [ ]:
tags = list("ab")
d3D.u.title = "some tags"
d3D.u.labels = tags
d3D

or more complex objects.

For instance here we use datetime.timedelta objects:

In [ ]:
from datetime import timedelta

start = timedelta(0)
times = [start + timedelta(seconds=x * 60) for x in range(6)]
d3D.t = None
d3D.t.labels = times
d3D.t.title = "time"
d3D

In this case, getting a coordinate that doesn't possess numerical data but labels,
will return the labels

In [ ]:
d3D.time

# More insight on coordinates

## Sharing coordinates between dimensions

Sometimes it is not necessary to have different coordinates for each axis. Some can be
shared between axes.

For example, if we have a square matrix with the same coordinate in the two
dimensions, the second dimension can
refer to the first. Here we create a square 2D dataset, using the `diag` method:

In [ ]:
nd = NDDataset.diag((3, 3, 2.5))
nd

and then we add the same coordinate for both dimensions

In [ ]:
coordx = Coord.arange(3)
nd.set_coordset(x=coordx, y="x")
nd

## Setting coordinates using `set_coordset`

Let's create 3 `Coord` objects to be used as coordinates for the 3 dimensions of the
previous d3D dataset.

In [ ]:
d3D.dims = ["t", "v", "u"]
s0, s1, s2 = d3D.shape
coord0 = Coord.linspace(10.0, 100.0, s0, units="m", title="distance")
coord1 = Coord.linspace(20.0, 25.0, s1, units="K", title="temperature")
coord2 = Coord.linspace(0.0, 1000.0, s2, units="hour", title="elapsed time")

### Syntax 1

In [ ]:
d3D.set_coordset(u=coord2, v=coord1, t=coord0)
d3D

### Syntax 2

In [ ]:
d3D.set_coordset({"u": coord2, "v": coord1, "t": coord0})
d3D

## Adding several coordinates to a single dimension
We can add several coordinates to the same dimension

In [ ]:
coord1b = Coord([1, 2, 3, 4], units="millitesla", title="magnetic field")

In [ ]:
d3D.set_coordset(u=coord2, v=[coord1, coord1b], t=coord0)
d3D

We can retrieve the various coordinates for a single dimension easily:

In [ ]:
d3D.v_1

## Math operations on coordinates
Arithmetic operations can be performed on single coordinates:

In [ ]:
d3D.u = d3D.u * 2
d3D.u

The ufunc numpy functions can also be applied, and will affect both the magnitude and
the units of the coordinates:

In [ ]:
d3D.u = 1.5 + np.sqrt(d3D.u)
d3D.u

A particularly frequent use case is to subtract the initial value from a coordinate. This can be done
directly with the `-` operator:

In [ ]:
d3D.u = d3D.u - d3D.u[0]
d3D.u

The operations above will generally *not* work on multiple coordinates, and
will raise an error if attempted:

In [ ]:
try:
    d3D.v = d3D.v - 1.5
except NotImplementedError as e:
    scp.error_(NotImplementedError, e)

Only subtraction between multiple coordinates is allowed, and will return a new `CoordSet` where each coordinate
has been subtracted:

In [ ]:
d3D.v = d3D.v - d3D.v[0]
d3D.v

It is always possible to carry out operations on a given coordinate
of a CoordSet. This must be done by accessing the coordinate by its name, e.g. `'temperature'` or `'_2'` for
the second coordinate of the `v` dimension:

In [ ]:
d3D.v["_2"] = d3D.v["_2"] + 5.0
d3D.v

## Summary of the coordinate setting syntax
Some additional information about coordinate setting syntax

**A.** First syntax (probably the safer because the name of the dimension is
specified, so this is less prone to
errors!)

In [ ]:
d3D.set_coordset(u=coord2, v=[coord1, coord1b], t=coord0)
# or equivalent
d3D.set_coordset(u=coord2, v=CoordSet(coord1, coord1b), t=coord0)
d3D

**B.** Second syntax assuming the coordinates are given in the order of the
dimensions.

Remember that we can check this order using the `dims` attribute of a NDDataset

In [ ]:
d3D.dims

In [ ]:
d3D.set_coordset((coord0, [coord1, coord1b], coord2))
# or equivalent
d3D.set_coordset(coord0, CoordSet(coord1, coord1b), coord2)
d3D

**C.** Third syntax (from a dictionary)

In [ ]:
d3D.set_coordset({"t": coord0, "u": coord2, "v": [coord1, coord1b]})
d3D

**D.** It is also possible to use directly the `CoordSet` property

In [ ]:
d3D.coordset = coord0, [coord1, coord1b], coord2
d3D

In [ ]:
d3D.coordset = {"t": coord0, "u": coord2, "v": [coord1, coord1b]}
d3D

In [ ]:
d3D.coordset = CoordSet(t=coord0, u=coord2, v=[coord1, coord1b])
d3D

<div class='alert alert-warning'>
<b>WARNING</b>
<p>
Do not use lists for setting multiple coordinates across different dimensions! Use tuples instead.
</p>
<p>
Lists have a special meaning in SpectroChemPy - they're used to set multiple coordinates for the same dimension.
</p>
</div>

This raises an error (lists have another meaning: they're used to set multiple coordinates
for the same dimension, as shown in examples A and B above):

In [ ]:
try:
    d3D.coordset = [coord0, coord1, coord2]
except ValueError:
    scp.error_(
        ValueError,
        "Coordinates must be of the same size for a dimension with multiple coordinates",
    )

This works: it uses a tuple `()`, not a list `[]`

In [ ]:
d3D.coordset = (
    coord0,
    coord1,
    coord2,
)  # equivalent to d3D.coordset = coord0, coord1, coord2
d3D

**E.** Setting the coordinates individually

Either a single coordinate

In [ ]:
d3D.u = coord2
d3D

or multiple coordinates for a single dimension

In [ ]:
d3D.v = [coord1, coord1b]
d3D

or using a CoordSet object.

In [ ]:
d3D.v = CoordSet(coord1, coord1b)
d3D

# Methods to create NDDataset

There are many ways to create `NDDataset` objects.

Let's first create 2 coordinate objects, for which we can define `labels` and `units`.
Note the use of the function `linspace` to generate the data.

In [ ]:
c0 = Coord.linspace(
    start=4000.0, stop=1000.0, num=5, labels=None, units="cm^-1", title="wavenumber"
)

In [ ]:
c1 = Coord.linspace(
    10.0, 40.0, 3, labels=["Cold", "RT", "Hot"], units="K", title="temperature"
)

The full coordset will be the following

In [ ]:
cs = CoordSet(c0, c1)
cs

Now we will generate the full dataset using the `fromfunction` method.
All needed information is passed as
parameters to the NDDataset constructor.

## Create a dataset from a function

In [ ]:
def func(x, y, extra):
    return x * y / extra

In [ ]:
ds = NDDataset.fromfunction(
    func,
    extra=100 * ur.cm**-1,  # extra arguments passed to the function
    coordset=cs,
    name="mydataset",
    title="absorbance",
    units=None,
)  # when None, units will be determined from the function results

ds.description = """Dataset example created for this tutorial.
It's a 2-D dataset"""

ds.author = "Blake & Mortimer"
ds

## Using numpy-like constructors of NDDatasets

In [ ]:
dz = NDDataset.zeros(
    (5, 3), coordset=cs, units="meters", title="Datasets with only zeros"
)

In [ ]:
do = NDDataset.ones(
    (5, 3), coordset=cs, units="kilograms", title="Datasets with only ones"
)

In [ ]:
df = NDDataset.full(
    (5, 3), fill_value=1.25, coordset=cs, units="radians", title="with only float=1.25"
)
df

As with numpy, it is also possible to take another dataset as a template:

In [ ]:
df = NDDataset.full_like(d3D, dtype="int", fill_value=2)
df

In [ ]:
nd = NDDataset.diag((3, 3, 2.5))
nd

## Copying existing NDDataset

To copy an existing dataset, this is as simple as:

In [ ]:
d3D_copy = d3D.copy()

or alternatively:

In [ ]:
d3D_copy_alt = d3D[:]

Finally, it is also possible to initialize a dataset using an existing one:

In [ ]:
d3Dduplicate = NDDataset(d3D, name=f"duplicate of {d3D.name}", units="absorbance")
d3Dduplicate

## Importing from external datasets

NDDatasets can be created from the importation of external data.

A **test** data folder contains some sample data for experimenting with features of
datasets.

In [ ]:
# let check if this directory exists and display its actual content:
datadir = scp.preferences.datadir
if datadir.exists():
    print(datadir.name)

Let's load grouped IR spectra acquired using OMNIC:

In [ ]:
nd = scp.read_omnic(datadir / "irdata/nh4y-activation.spg")
scp.preferences.reset()
_ = nd.plot()

Even if we do not specify the **datadir**, the application first looks in the
default directory.

Now, lets load a NMR dataset (in the Bruker format).

Requires the official ``spectrochempy-nmr`` plugin.
Install with: ``python -m pip install "spectrochempy[nmr]"``.

In [ ]:
path = datadir / "nmrdata" / "bruker" / "tests" / "nmr" / "topspin_1d"

# load the data directly (no need to create the dataset first)
nd2 = scp.nmr.read(path, expno=1, remove_digital_filter=True)

# view it...
nd2.x.to("s")

ax = nd2.plot()